# Curso de Métodos Numéricos

## Práctica 7: Interpolación Polinomial y Splines Cúbicos


--- 
### Teoría: Polinomio de Interpolación de Newton (Diferencias Divididas)
Dado un conjunto de $n+1$ puntos distintos $(x_0, y_0), (x_1, y_1), \dots, (x_n, y_n)$, el polinomio de interpolación en forma de Newton se expresa como:
$$P_n(x) = f[x_0] + f[x_0, x_1](x - x_0) + f[x_0, x_1, x_2](x - x_0)(x - x_1) + \dots + f[x_0, x_1, \dots, x_n] \prod_{j=0}^{n-1} (x - x_j)$$

Donde las **diferencias divididas** se definen recursivamente mediante:
- **Orden 0:**  
  $$f[x_i] = y_i$$
- **Orden 1:**  
  $$f[x_i, x_{i+1}] = \frac{f[x_{i+1}] - f[x_i]}{x_{i+1} - x_i}$$
- **Orden $k$ (General):**  
  $$f[x_i, x_{i+1}, \dots, x_{i+k}] = \frac{f[x_{i+1}, \dots, x_{i+k}] - f[x_i, \dots, x_{i+k-1}]}{x_{i+k} - x_i}$$

En esta práctica denotaremos los coeficientes del polinomio de Newton como: $c_k = f[x_0, x_1, \dots, x_k]$.

--- 

## Problema 1: Interpolación de Newton con Diferencias Divididas

### Actividades:
1. Define una función `diferencias_divididas(x, y)` que reciba el vector de nodos $x$, el vector de valores $y$ de la función y que regrese un vector con los valores de las diferencias divididas $c_k = f[x_0, x_1, \dots, x_k]$.
2. Consideremos la función $f(x) = \cos(x)$ en el intervalo $[0, \pi]$. Se seleccionan $4$ nodos equiespaciados ($n=3$):
$$x_0 = 0, \quad x_1 = \frac{\pi}{3}, \quad x_2 = \frac{2\pi}{3}, \quad x_3 = \pi$$
Calcule los valores exactos $y_i = \cos(x_i)$ para $i=0,1,2,3$, luego usa tu función `diferencias_divididas(x, y)` para mostrar que el polinomio es 
$$
P_3(x) = 1 - \frac{3}{2\pi}x - \frac{9}{4\pi^2}x\left(x - \frac{\pi}{3}\right) + \frac{9}{2\pi^3}x\left(x - \frac{\pi}{3}\right)\left(x - \frac{2\pi}{3}\right)
$$ 
En forma simplificada, dicho polinomio es:
$$
P_3(x) = \frac{9}{2\pi^3} x^3 - \frac{27}{4\pi^2} x^2 + \frac{1}{4\pi} x + 1 = 0.145132 x^3 - 0.683918 x^2 + 0.079577 x + 1
$$
3. Grafique en una misma figura la función exacta $f(x)$, el polinomio $P_3(x)$ y los $4$ nodos (marcalos con una estrella) de interpolación en $[0, \pi]$.

In [1]:
pass

--- 
### Teoría: Spline Cúbico Natural
Dado un conjunto de $n+1$ nodos ordenados $x_0 < x_1 < \dots < x_n$ y sus respectivos valores $y_i$, se desea encontrar una función $S(x)$ compuesta por $n$ polinomios cúbicos por tramos $S_i(x)$ definidos en $[x_i, x_{i+1}]$ for $i=0, 1, \dots, n-1$:
$$S_i(x) = a_i + b_i(x - x_i) + c_i(x - x_i)^2 + d_i(x - x_i)^3, \quad x \in [x_i, x_{i+1}]$$

**Condiciones de continuidad y suavidad:**
1. **Interpolación:** $S_i(x_i) = y_i$ y $S_i(x_{i+1}) = y_{i+1}$.
2. **Continuidad de $S'(x)$:** $S_{i-1}'(x_i) = S_i'(x_i)$ para $i=1, \dots, n-1$.
3. **Continuidad de $S''(x)$:** $S_{i-1}''(x_i) = S_i''(x_i)$ para $i=1, \dots, n-1$.
4. **Condiciones Naturales de Frontera:** La curvatura en los extremos se anula:
   $$S''(x_0) = 0 \implies c_0 = 0$$
   $$S''(x_n) = 0 \implies c_n = 0$$

Sea $h_i = x_{i+1} - x_i$. Los coeficientes $c_i$ se determinan resolviendo el sistema lineal tridiagonal:
$$h_{i-1}c_{i-1} + 2(h_{i-1} + h_i)c_i + h_i c_{i+1} = 3\left( \frac{y_{i+1} - y_i}{h_i} - \frac{y_i - y_{i-1}}{h_{i-1}} \right), \quad i=1, 2, \dots, n-1$$
que en términos de los momentos $M_i = 2c_i$ equivale a 
$$h_{i-1}M_{i-1} + 2(h_{i-1} + h_i)M_i + h_i M_{i+1} = 6\left( \frac{y_{i+1} - y_i}{h_i} - \frac{y_i - y_{i-1}}{h_{i-1}} \right), \quad i=1, 2, \dots, n-1$$

Una vez determinados los momentos $M_i$, el polinomio cúbico en cada subintervalo $[x_i, x_{i+1}]$ se expresa como:
$$
S_i(x) = a_i + b_i(x - x_i) + c_i(x - x_i)^2 + d_i(x - x_i)^3
$$
con coeficientes dados por:
$$
a_i = y_i, \quad c_i = \frac{M_i}{2}, \quad d_i = \frac{M_{i+1} - M_i}{6 h_i}, \quad b_i = \frac{y_{i+1} - y_i}{h_i} - \frac{h_i(2 M_i + M_{i+1})}{6}
$$

--- 
## Problema 2: Spline Cúbico Natural con `scipy`

Consideremos las siguientes tres funciones:

| | Función | Intervalo | Nodos equiespaciados |
|---|---|---|---|
| (a) | $f_1(x) = \sin(x)$ | $[0, 2\pi]$ | $5$ nodos ($n=4$) |
| (b) | $f_2(x) = e^{-x^2}$ | $[-2, 2]$ | $5$ nodos ($n=4$) |
| (c) | $f_3(x) = \ln(1 + x)$ | $[0, 4]$ | $4$ nodos ($n=3$) |

### Actividades:
1. La librería `scipy` se puede utilizar para construir el spline cúbico interpolador de un conjunto de nodos. Investiga como usar `scipy` para obtener el spline cúbico natural $S(x)$ de cada función en el intervalo pedido.
2. Grafica para cada caso la función $f(x)$, el spline interpolador $S(x)$ y los nodos de interpolación (márcalos con una estrella) en su intervalo correspondiente.`

Ps. La teoría del markdown anterior no es necesaria pero siempre es bueno saberla =) 


In [1]:
pass